# moodwall-ko-v0 training
Runtime: T4 GPU

### 1. GPU

In [ ]:
!nvidia-smi

### 2. Install

In [ ]:
!pip install -q transformers scikit-learn

### 3. Upload seed_data.csv

In [ ]:
from google.colab import files
files.upload()

### 4. Data

In [ ]:
import pandas as pd

seed = pd.read_csv("seed_data.csv", encoding="utf-8-sig")

UNSMILE = "https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_train_v1.0.tsv"
un = pd.read_csv(UNSMILE, sep="\t")
un = un[(un["clean"] == 0) & (un["문장"].str.len() <= 40)].sample(40, random_state=42)

df = pd.concat([
    seed[["text", "label"]],
    pd.DataFrame({"text": un["문장"].values, "label": "inappropriate"}),
]).drop_duplicates("text").reset_index(drop=True)

print(len(df))
df["label"].value_counts()

### 5. Model

In [ ]:
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABELS = ["congrats", "joy", "cheer", "excited", "tired", "sad", "angry", "inappropriate"]
KO = dict(zip(LABELS, ["축하", "기쁨", "응원", "설렘", "피곤", "슬픔", "화남", "부적절"]))
label2id = {l: i for i, l in enumerate(LABELS)}

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# held out for comparing model versions
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)
print(len(train_df), len(test_df))

MODEL_NAME = "beomi/KcELECTRA-base-v2022"
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label={i: KO[l] for i, l in enumerate(LABELS)},
    label2id={KO[l]: i for i, l in enumerate(LABELS)},
).to(device)
device

### 6. Train

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_ds(d):
    enc = tok(list(d["text"]), padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    y = torch.tensor([label2id[l] for l in d["label"]])
    return TensorDataset(enc["input_ids"], enc["attention_mask"], y)

loader = DataLoader(make_ds(train_df), batch_size=16, shuffle=True)
optim = torch.optim.AdamW(model.parameters(), lr=5e-5)
EPOCHS = 8

model.train()
for ep in range(EPOCHS):
    total = 0
    for ids, mask, y in loader:
        out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
        out.loss.backward()
        optim.step()
        optim.zero_grad()
        total += out.loss.item()
    print(f"epoch {ep + 1} loss {total / len(loader):.4f}")

### 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

model.eval()
preds, bad_probs = [], []
with torch.no_grad():
    for i in range(0, len(test_df), 32):
        batch = test_df["text"].iloc[i:i + 32].tolist()
        enc = tok(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
        p = torch.softmax(model(**enc).logits, -1)
        preds += p.argmax(-1).tolist()
        bad_probs += p[:, label2id["inappropriate"]].tolist()

y_true = [label2id[l] for l in test_df["label"]]
names = [KO[l] for l in LABELS]

print("acc", round(accuracy_score(y_true, preds), 3))
print(classification_report(y_true, preds, labels=list(range(len(LABELS))), target_names=names, zero_division=0))

result = test_df.assign(true=test_df["label"].map(KO), pred=[names[p] for p in preds])
print(pd.crosstab(result["true"], result["pred"]))
result[result["true"] != result["pred"]]

### 7-2. Unsafe threshold

In [ ]:
is_bad = np.array(y_true) == label2id["inappropriate"]
bp = np.array(bad_probs)

for th in [0.1, 0.2, 0.3, 0.4, 0.5]:
    hold = bp >= th
    recall = (hold & is_bad).sum() / is_bad.sum()
    fp = (hold & ~is_bad).sum() / (~is_bad).sum()
    print(f"{th}: recall {recall:.0%}, false hold {fp:.0%}")

### 8. Try

In [ ]:
def predict(text):
    enc = tok(text, truncation=True, max_length=64, return_tensors="pt").to(device)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, -1)[0]
    i = p.argmax().item()
    return model.config.id2label[i], round(p[i].item(), 3)

for s in ["지수야 생일 ㅊㅋ!!", "내일 축제 개기대됨", "밤샘 과제 너무 지친다", "학식 왜 맨날 이럼", "기말 다들 화이팅"]:
    print(s, predict(s))

### 9. Save to Drive

In [ ]:
model.save_pretrained("moodwall_model")
tok.save_pretrained("moodwall_model")
test_df.to_csv("test_set_v0.csv", index=False, encoding="utf-8-sig")

from google.colab import drive
drive.mount("/content/drive")
!zip -rq moodwall_model.zip moodwall_model
!cp moodwall_model.zip test_set_v0.csv /content/drive/MyDrive/